In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [21]:
def minmax_scale_to_range(x, new_min, new_max):
    """Min-max scale 1D array x to [new_min, new_max]."""
    xmin, xmax = np.nanmin(x), np.nanmax(x)
    if np.isclose(xmax, xmin):
        return np.full_like(x, (new_min + new_max) / 2.0)
    return ( (x - xmin) / (xmax - xmin) ) * (new_max - new_min) + new_min

def generate_chain_data_scaled(N=20000,
                               confounding_strength=0.8,
                               noise_std=1.0,
                               s1_range=(0.0, 2.0),
                               s2_range=(1.0, 4.0),
                               round_decimals=2):
    """
    Generate minimal synthetic dataset for two-stage pipeline.
    Columns: Z1,Z2,Z3, T1,T2, s1_m11,s1_m12, s2_m21,s2_m22
    Ensures stage1 models are in s1_range and stage2 models in s2_range.
    """
    # ---- Protected attributes ----
    Z1 = np.random.choice([0,1,2], size=N, p=[0.45,0.45,0.10])
    Z2 = np.random.choice([0,1], size=N, p=[0.5,0.5])
    Z3 = np.random.choice([0,1,2], size=N, p=[0.3,0.5,0.2])
    Z1n, Z2n, Z3n = Z1.astype(float), Z2.astype(float), Z3.astype(float)

    # ---- Treatments (confounded) ----
    logit_T1 = -0.5 + confounding_strength * (0.4*Z1n + 0.6*Z2n + 0.2*Z3n)
    p_T1 = 1 / (1 + np.exp(-logit_T1))
    T1 = (np.random.rand(N) < p_T1).astype(int)

    logit_T2 = -0.3 + confounding_strength * (0.3*Z1n + 0.4*Z2n + 0.3*Z3n)
    p_T2 = 1 / (1 + np.exp(-logit_T2))
    T2 = (np.random.rand(N) < p_T2).astype(int)

    # ---- Raw model outputs (before scaling) ----
    s1_m11_raw = 0.8*Z1n + 0.5*Z2n + 0.3*Z3n + np.random.normal(0, noise_std, N)
    s1_m12_raw = 1.0*Z1n - 0.4*Z2n + 0.2*Z3n + np.random.normal(0, noise_std, N)

    # Stage2 depends on stage1's raw outputs and T1 for realism
    s2_m21_raw = 0.6*s1_m11_raw + 0.4*T1 + 0.3*Z3n + np.random.normal(0, noise_std, N)
    s2_m22_raw = 0.9*s1_m12_raw - 0.2*T1 + 0.5*Z1n + np.random.normal(0, noise_std, N)

    # ---- Scale per-model to desired ranges ----
    s1_m11 = minmax_scale_to_range(s1_m11_raw, s1_range[0], s1_range[1])
    s1_m12 = minmax_scale_to_range(s1_m12_raw, s1_range[0], s1_range[1])

    s2_m21 = minmax_scale_to_range(s2_m21_raw, s2_range[0], s2_range[1])
    s2_m22 = minmax_scale_to_range(s2_m22_raw, s2_range[0], s2_range[1])

    # ---- Assemble DataFrame and round ----
    df = pd.DataFrame({
        'Z1': Z1,
        'Z2': Z2,
        'Z3': Z3,
        'T1': T1,
        'T2': T2,
        's1_m11': s1_m11,
        's1_m12': s1_m12,
        's2_m21': s2_m21,
        's2_m22': s2_m22
    })

    # Round numeric columns
    float_cols = df.select_dtypes(include=[np.floating]).columns.tolist()
    df[float_cols] = df[float_cols].round(round_decimals)

    return df


In [23]:
# ---------------- Generate and save ----------------
df = generate_chain_data_scaled(N=30000,
                                confounding_strength=0.8,
                                noise_std=1.2,
                                s1_range=(0.0, 2.0),
                                s2_range=(1.0, 4.0),
                                round_decimals=2)

out_path = Path('../data/input/synthetic_chain.csv')
df.to_csv(out_path, index=False)
print(f"Wrote {out_path} with shape {df.shape}")

# Quick verification of ranges (min/max)
for col in ['s1_m11','s1_m12']:
    print(f"{col} min/max: {df[col].min():.2f} / {df[col].max():.2f}")
for col in ['s2_m21','s2_m22']:
    print(f"{col} min/max: {df[col].min():.2f} / {df[col].max():.2f}")

# Show a few rows
print("\nSample rows:")
print(df.head())


Wrote ../data/input/synthetic_chain.csv with shape (30000, 9)
s1_m11 min/max: 0.00 / 2.00
s1_m12 min/max: 0.00 / 2.00
s2_m21 min/max: 1.00 / 4.00
s2_m22 min/max: 1.00 / 4.00

Sample rows:
   Z1  Z2  Z3  T1  T2  s1_m11  s1_m12  s2_m21  s2_m22
0   0   0   1   1   0    0.68    1.03    2.09    2.37
1   1   1   1   1   1    0.97    0.98    2.44    2.63
2   1   1   1   1   0    1.13    1.24    3.07    2.36
3   0   1   0   0   1    0.93    0.54    1.82    1.18
4   0   1   1   0   1    0.86    0.80    2.57    2.14
